结构化输出的4种策略

1 ProviderStrategy 用于支持原生结构化输出的模型

In [1]:
from typing import Literal
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain.agents import create_agent
import os
from pydantic import BaseModel, Field
from rich import print as rprint
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
load_dotenv(override=True)
my_api_key=os.getenv("API_KEY")
my_base_url=os.getenv("BASE_URL")
#1 初始化大模型
myModel= init_chat_model(
    model="qwen3.8-max",
    model_provider="openai",
    api_key=my_api_key,
    base_url=my_base_url,
    timeout=30,
    max_retries=3,
)
class ContractInfo(BaseModel):
    """
    提取用户的联系方式
    """
    name:str=Field(description="用户姓名")
    email:str=Field(description="用户邮箱号")
    phone:str=Field(description="用户手机号")
#3 初始化我的agent
myAgent=create_agent(
    model=myModel,
    tools=[],
    name="agent01",
    # system_prompt="你是1个专业的天气预报小助手,当有人问你关于天气问题的时候，你会调用工具get_weather来帮你完成查询任务。"
    system_prompt=SystemMessage(content="你是1个专业的信息处理小助手,当有人问你关于信息处理问题的时候，你会帮助询问者完成信息处理任务。"),
    response_format=ProviderStrategy(ContractInfo),
)
#4 调用agent
response=myAgent.invoke(
    {
        # 严格校验，使用message少了1个s就会报错

        "messages":
            [
                HumanMessage("请你提取一下该员工的联系信息：小明的邮箱是1414227322@qq.com,手机是18788483333")
        ]
    }
)
rprint(response)
# print(response)

{
    'messages': [
        HumanMessage(
            content='请你提取一下该员工的联系信息：小明的邮箱是1414227322@qq.com,手机是18788483333',
            additional_kwargs={},
            response_metadata={},
            id='f253cd70-1d43-41e4-b2e4-1b8456115317'
        ),
        AIMessage(
            content='{"name": "小明", "email": "1414227322@qq.com", "phone": "18788483333"}',
            additional_kwargs={'parsed': None, 'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 163,
                    'prompt_tokens': 238,
                    'total_tokens': 401,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 121,
                        'rejected_prediction_tokens': None,
                        'text_tokens': 163
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 0,
                        'image_tokens': None,
                        'text_tokens': 238
                    }
                },
                'model_provider': 'openai',
                'model_name': 'qwen3.8-max',
                'system_fingerprint': None,
                'id': 'chatcmpl-e620cf60-8466-9c7e-a6c9-d8564cc1925e',
                'finish_reason': 'stop',
                'logprobs': None
            },
            name='agent01',
            id='lc_run--01a0a92d-481d-7b41-a7d1-681d13a42c45-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 238,
                'output_tokens': 163,
                'total_tokens': 401,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {'reasoning': 121}
            }
        )
    ],
    'structured_response': ContractInfo(name='小明', email='1414227322@qq.com', phone='18788483333')
}

2 ToolStrategy如果不支持原生结构化输出的模型，Langchain采用ToolStrategy工具调用方式来实现

In [9]:
from typing import Literal
from langchain.agents.structured_output import ProviderStrategy,ToolStrategy
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain.agents import create_agent
import os
from pydantic import BaseModel, Field
from rich import print as rprint
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
load_dotenv(override=True)
my_api_key=os.getenv("API_KEY")
my_base_url=os.getenv("BASE_URL")
#1 初始化大模型
myModel= init_chat_model(
    model="qwen3.8-max",
    model_provider="openai",
    api_key=my_api_key,
    base_url=my_base_url,
    timeout=30,
    max_retries=3,
    model_kwargs={"reasoning_effort":"none"}#禁用思考模式
)
class ContractInfo(BaseModel):
    """
    提取用户的联系方式
    """
    name:str=Field(description="用户姓名")
    email:str=Field(description="用户邮箱号")
    phone:str=Field(description="用户手机号")
#3 初始化我的agent
myAgent=create_agent(
    model=myModel,
    name="agent02",
    tools=[],
    # system_prompt="你是1个专业的天气预报小助手,当有人问你关于天气问题的时候，你会调用工具get_weather来帮你完成查询任务。"
    system_prompt=SystemMessage(content="你是1个专业的信息处理小助手,当有人问你关于信息处理问题的时候，你会帮助询问者完成信息处理任务。"),
    response_format=ToolStrategy(ContractInfo),
)
#4 调用agent
response=myAgent.invoke(
    {
        # 严格校验，使用message少了1个s就会报错

        "messages":
            [
                HumanMessage("请你提取一下该员工的联系信息：小明的邮箱是1414227322@qq.com,手机是18788483333")
        ]
    }
)
rprint(response)
# print(response)

C:\Users\Administrator\.conda\envs\langchain1.2\Lib\site-packages\langchain\chat_models\base.py:516: UserWarning: Parameters {'reasoning_effort'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  return _init_chat_model_helper(


{
    'messages': [
        HumanMessage(
            content='请你提取一下该员工的联系信息：小明的邮箱是1414227322@qq.com,手机是18788483333',
            additional_kwargs={},
            response_metadata={},
            id='b2e69dcf-c4da-46e3-b563-7fb8d1eb15bb'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 66,
                    'prompt_tokens': 384,
                    'total_tokens': 450,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 0,
                        'image_tokens': None,
                        'text_tokens': 384
                    }
                },
                'model_provider': 'openai',
                'model_name': 'qwen3.8-max',
                'system_fingerprint': None,
                'id': 'chatcmpl-e412e3b5-ec3b-95ed-adf3-db530ac98002',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            name='agent02',
            id='lc_run--01a0a93a-c829-7530-a270-dc738112004f-0',
            tool_calls=[
                {
                    'name': 'ContractInfo',
                    'args': {'name': '小明', 'email': '1414227322@qq.com', 'phone': '18788483333'},
                    'id': 'call_70d0907504394df5a7ea6bd5',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 384,
                'output_tokens': 66,
                'total_tokens': 450,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content="Returning structured response: name='小明' email='1414227322@qq.com' phone='18788483333'",
            name='ContractInfo',
            id='3edd6386-4b7e-48ec-848a-14f3b0de7c2e',
            tool_call_id='call_70d0907504394df5a7ea6bd5'
        )
    ],
    'structured_response': ContractInfo(name='小明', email='1414227322@qq.com', phone='18788483333')
}

3 type/AutoStrategy 当直接传入1个定义类型的时候，langchain会自动包装为AutoStrategy，触发自动选择策略，如果大模型支持原生结构化输出，则使用ProviderStrategy,否则使用ToolStrategy

In [11]:
from typing import Literal
from langchain.agents.structured_output import ProviderStrategy,ToolStrategy,AutoStrategy
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain.agents import create_agent
import os
from pydantic import BaseModel, Field
from rich import print as rprint
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
load_dotenv(override=True)
my_api_key=os.getenv("API_KEY")
my_base_url=os.getenv("BASE_URL")
#1 初始化大模型
myModel= init_chat_model(
    model="qwen3.8-max",
    model_provider="openai",
    api_key=my_api_key,
    base_url=my_base_url,
    timeout=30,
    max_retries=3,
    model_kwargs={"reasoning_effort":"none"}#禁用思考模式
)
class ContractInfo(BaseModel):
    """
    提取用户的联系方式
    """
    name:str=Field(description="用户姓名")
    email:str=Field(description="用户邮箱号")
    phone:str=Field(description="用户手机号")
#3 初始化我的agent
myAgent=create_agent(
    model=myModel,
    name="agent02",
    tools=[],
    # system_prompt="你是1个专业的天气预报小助手,当有人问你关于天气问题的时候，你会调用工具get_weather来帮你完成查询任务。"
    system_prompt=SystemMessage(content="你是1个专业的信息处理小助手,当有人问你关于信息处理问题的时候，你会帮助询问者完成信息处理任务。"),
    response_format=AutoStrategy(ContractInfo),
)
#4 调用agent
response=myAgent.invoke(
    {
        # 严格校验，使用message少了1个s就会报错

        "messages":
            [
                HumanMessage("请你提取一下该员工的联系信息：小明的邮箱是1414227322@qq.com,手机是18788483333")
        ]
    }
)
rprint(response)
# print(response)

C:\Users\Administrator\.conda\envs\langchain1.2\Lib\site-packages\langchain\chat_models\base.py:516: UserWarning: Parameters {'reasoning_effort'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  return _init_chat_model_helper(


{
    'messages': [
        HumanMessage(
            content='请你提取一下该员工的联系信息：小明的邮箱是1414227322@qq.com,手机是18788483333',
            additional_kwargs={},
            response_metadata={},
            id='32bc5ef7-0fed-4018-b8af-27bb3a0dd31b'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 66,
                    'prompt_tokens': 384,
                    'total_tokens': 450,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 0,
                        'image_tokens': None,
                        'text_tokens': 384
                    }
                },
                'model_provider': 'openai',
                'model_name': 'qwen3.8-max',
                'system_fingerprint': None,
                'id': 'chatcmpl-1c9ff0bd-0256-9787-a37c-ff2e63590a89',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            name='agent02',
            id='lc_run--01a0a93e-fd9b-7802-aa9c-76766ef21fb5-0',
            tool_calls=[
                {
                    'name': 'ContractInfo',
                    'args': {'name': '小明', 'email': '1414227322@qq.com', 'phone': '18788483333'},
                    'id': 'call_c0c157a908694f8bae8f2aa2',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 384,
                'output_tokens': 66,
                'total_tokens': 450,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content="Returning structured response: name='小明' email='1414227322@qq.com' phone='18788483333'",
            name='ContractInfo',
            id='085f8cc8-3da7-42a2-983e-e74393d6615c',
            tool_call_id='call_c0c157a908694f8bae8f2aa2'
        )
    ],
    'structured_response': ContractInfo(name='小明', email='1414227322@qq.com', phone='18788483333')
}

In [ ]:
4 None不以结构化输出，使用自然语言输出

最推荐使用ToolStrategy，大部门模型都支持了